In [1]:
!pip install git+https://github.com/lightshing/wildlife-datasets@develop
!pip install git+https://github.com/lightshing/wildlife-tools

  Cloning https://github.com/lightshing/wildlife-datasets (to revision develop) to /tmp/pip-req-build-5ogxnm7k
  Running command git clone --filter=blob:none --quiet https://github.com/lightshing/wildlife-datasets /tmp/pip-req-build-5ogxnm7k
  Running command git checkout -b develop --track origin/develop
  Switched to a new branch 'develop'
  Branch 'develop' set up to track remote branch 'develop' from 'origin'.
  Resolved https://github.com/lightshing/wildlife-datasets to commit 7f50c01c67b5d719e39880dcb2ea8317714f2d3c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 6.1 MB/s eta 0:00:00
  Created wheel for wildlife-datasets: filename=wildlife_datasets-1.0.5-py3-none-any.whl size=88545 sha256=d89e7b0e413fde279df820af4b58340380a8543a34acdbae3bd35112c1cf5abc
  Stored in directory: /tmp/pip-ephem-wheel-cache-4jtm8o5e/wheels/71/21/92/48eb881ac5

In [2]:
import os
import numpy as np
import pandas as pd
import timm
import torchvision.transforms as T
import imgaug as ia
import imgaug.augmenters as iaa
import torch
from torch import nn
from torch import optim
import tqdm
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import os
from datetime import datetime
from wildlife_datasets.datasets import AnimalCLEF2025
from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity
from wildlife_datasets import datasets, splits
from wildlife_tools.tools import set_seed
from wildlife_tools.train import BasicTrainer
from sklearn.preprocessing import LabelEncoder



from torch.utils.data import DataLoader


def create_sample_submission(dataset_query, predictions, file_name='sample_submission.csv'):
    df = pd.DataFrame({
        'image_id': dataset_query.metadata['image_id'],
        'identity': predictions
    })
    df.to_csv(file_name, index=False)

2025-05-12 18:59:50.294464: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747076390.487763      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747076390.546046      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
root = '/kaggle/input/animal-clef-2025'
out_root = '/kaggle/working'
checkpoint_out_path = '/kaggle/working/model_checkpoints'

In [4]:
set_split_seed = 998
training_seed = 669
ia.seed(366)

In [5]:
current_time = datetime.now().strftime('%b%d_%H-%M-%S')
log_dir = os.path.join(out_root, current_time)
os.makedirs(log_dir, exist_ok=True)

writer = SummaryWriter(log_dir=log_dir)

In [6]:
name = 'hf-hub:BVRA/MegaDescriptor-L-384'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mega = timm.create_model(name, num_classes=0, pretrained=True)
print(device)
mega = mega.to(device)

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

cuda


In [7]:
transform_mega_ori = timm.data.create_transform(
    **timm.data.resolve_data_config(mega.pretrained_cfg)
)
transform_mega_ori

Compose(
    Resize(size=426, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(384, 384))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)

In [8]:
transform_base_aug = T.Compose([
    iaa.Sequential([
        iaa.Sometimes(0.7, iaa.MultiplyBrightness((0.8, 1.3))),
        iaa.Sometimes(0.3, iaa.GammaContrast((0.7, 1.8))),
        iaa.Sometimes(0.8, iaa.CLAHE(clip_limit=(1, 6), tile_grid_size_px=(8, 8))),
        iaa.Sometimes(0.2, iaa.AllChannelsCLAHE(clip_limit=(1, 6), tile_grid_size_px=(4, 8))),
        
        iaa.Sometimes(0.3, iaa.AddToHueAndSaturation((-13, 13))),
        iaa.Sometimes(0.6,
            iaa.WithColorspace(
                to_colorspace="HSV",
                from_colorspace="RGB",
                children=iaa.WithChannels(
                    0,
                    iaa.Add((-10, 10))
                )
            )
        ),
        
        iaa.Sometimes(0.6,iaa.AdditivePoissonNoise(lam=(0, 5))),
        
        iaa.Sometimes(0.3,
            iaa.Affine(
                translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
                mode='constant',
                cval=0
            )
        ),
        iaa.Sometimes(0.4,
            iaa.Affine(
                rotate=(-30, 30),
                mode='constant',
                cval=0
            )
        ),
        iaa.Sometimes(0.6,
            iaa.Affine(
                scale={"x": (0.8, 1.2), "y": (0.8, 1.2)},
                mode='constant',
                cval=0
            )
        ),
        iaa.Sometimes(0.8,iaa.Fliplr(0.5)),
        
        iaa.Sometimes(0.1, iaa.GaussianBlur(sigma=(0.0, 1.0))),
        iaa.Sometimes(0.3, iaa.Sharpen(alpha=(0.0, 0.5), lightness=(0.6, 1.4)))
    ], random_order=True).augment_image,
    ])

In [9]:
transform = T.Compose([
    T.ToTensor(),
    *transform_mega_ori.transforms
    ])

In [10]:
transform_aug = T.Compose([
    *transform_base_aug.transforms,
    T.ToTensor(),
    *transform_mega_ori.transforms
    ])

In [11]:
# Loading the dataset
dataset = AnimalCLEF2025(root, transform=transform_aug, load_label=True)
dataset_database = dataset.get_subset(dataset.metadata['split'] == 'database')
dataset_query = dataset.get_subset(dataset.metadata['split'] == 'query')
n_query = len(dataset_query)

In [12]:
dataset.metadata[['dataset', 'split']].value_counts()

dataset           split   
SeaTurtleID2022   database    8729
LynxID2025        database    2957
SalamanderID2025  database    1388
LynxID2025        query        946
SalamanderID2025  query        689
SeaTurtleID2022   query        500
Name: count, dtype: int64

### 分为3个set, open close
train/test/thresholdSet

In [13]:
df_database = dataset_database.df
df_database.dataset.value_counts()

dataset
SeaTurtleID2022     8729
LynxID2025          2957
SalamanderID2025    1388
Name: count, dtype: int64

In [14]:
openSplitter = splits.OpenSetSplit(0.9, 0.1, seed=set_split_seed)
for idx_train, idx_test in openSplitter.split(df_database):
    splits.analyze_split(df_database, idx_train, idx_test)

Split: time-unaware open-set
Samples: train/test/unassigned/total = 11048/2026/0/13074
Classes: train/test/unassigned/total = 965/833/0/1102
Samples: train only/test only        = 269/1330
Classes: train only/test only/joint  = 269/137/696

Fraction of train set     = 84.50%
Fraction of test set only = 10.17%


In [15]:
df_train, df_test = df_database.loc[idx_train], df_database.loc[idx_test]
print(len(df_train))
print(len(df_test))

11048
2026


In [16]:
trainSet = dataset_database.get_subset(df_train.index)
thresholdSet = dataset_database.get_subset(df_test.index)

#### 标签转换为数字

In [17]:
df_database_training = trainSet.df

df_database_training['name'] = df_database_training['identity'].copy()

encoder = LabelEncoder()
df_database_training['identity'] = encoder.fit_transform(df_database_training['identity'])

trainSet.df = df_database_training

In [18]:
df_database_training.to_csv("/kaggle/working/mm.csv", index=False, encoding='utf-8-sig')

In [19]:
df_database_training.dataset.value_counts()

dataset
SeaTurtleID2022     7342
LynxID2025          2718
SalamanderID2025     988
Name: count, dtype: int64

In [20]:
closeSplitter = splits.ClosedSetSplit(0.9, seed=set_split_seed)
for idx_train, idx_test in closeSplitter.split(df_database_training):
    splits.analyze_split(df_database_training, idx_train, idx_test)

Split: time-unaware closed-set
Samples: train/test/unassigned/total = 9820/1228/0/11048
Classes: train/test/unassigned/total = 965/596/0/965
Samples: train only/test only        = 369/0
Classes: train only/test only/joint  = 369/0/596

Fraction of train set     = 88.88%
Fraction of test set only = 0.00%


In [21]:
df_training, df_val = df_database_training.loc[idx_train], df_database_training.loc[idx_test]
print(len(df_training))
print(len(df_val))

9820
1228


In [22]:
training = trainSet.get_subset(df_training.index)
test = trainSet.get_subset(df_val.index)

In [23]:
training.df.dataset.value_counts()

dataset
SeaTurtleID2022     6554
LynxID2025          2438
SalamanderID2025     828
Name: count, dtype: int64

In [24]:
df = training.df

identity_counts = df['identity'].value_counts()


rows_to_append = []
for identity, count in identity_counts.items():
    if count < 5:
        rows = df[df['identity'] == identity]
        num_to_add = 5 - count
        repeated_rows = pd.concat([rows]*num_to_add, ignore_index=True).iloc[:num_to_add]
        rows_to_append.append(repeated_rows)

if rows_to_append:
    df = pd.concat([df] + rows_to_append, ignore_index=True)


training.df = df

In [25]:
training.df.dataset.value_counts()

dataset
SeaTurtleID2022     6771
SalamanderID2025    2603
LynxID2025          2506
Name: count, dtype: int64

### 模型预处理

In [26]:
print(mega)

SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
  )
  (layers): Sequential(
    (0): SwinTransformerStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=192, out_features=576, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=192, out_features=192, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path1): Identity()
          (norm2): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=192, out_features=768, bias=True)
            (act): GELU(approximate='none')
            (

In [27]:
mega.head.drop = nn.Dropout(p=0.33)
mega.head.fc = nn.Linear(1536, 965)

In [28]:
for name, param in mega.named_parameters():
    param.requires_grad = False
    if "norm.weight" == name:
        param.requires_grad = True
    if "norm.bias" == name:
        param.requires_grad = True
    if "layers.3" in name:
        param.requires_grad = True
    if "layers.2.blocks" in name:
        param.requires_grad = True
    if "head.fc" in name:
        param.requires_grad = True
for name, param in mega.named_parameters():
    print(f"{name}: {param.requires_grad}")

patch_embed.proj.weight: False
patch_embed.proj.bias: False
patch_embed.norm.weight: False
patch_embed.norm.bias: False
layers.0.blocks.0.norm1.weight: False
layers.0.blocks.0.norm1.bias: False
layers.0.blocks.0.attn.relative_position_bias_table: False
layers.0.blocks.0.attn.qkv.weight: False
layers.0.blocks.0.attn.qkv.bias: False
layers.0.blocks.0.attn.proj.weight: False
layers.0.blocks.0.attn.proj.bias: False
layers.0.blocks.0.norm2.weight: False
layers.0.blocks.0.norm2.bias: False
layers.0.blocks.0.mlp.fc1.weight: False
layers.0.blocks.0.mlp.fc1.bias: False
layers.0.blocks.0.mlp.fc2.weight: False
layers.0.blocks.0.mlp.fc2.bias: False
layers.0.blocks.1.norm1.weight: False
layers.0.blocks.1.norm1.bias: False
layers.0.blocks.1.attn.relative_position_bias_table: False
layers.0.blocks.1.attn.qkv.weight: False
layers.0.blocks.1.attn.qkv.bias: False
layers.0.blocks.1.attn.proj.weight: False
layers.0.blocks.1.attn.proj.bias: False
layers.0.blocks.1.norm2.weight: False
layers.0.blocks.1.norm

### 测试输出

In [29]:
mega.to(device)
test_input = torch.ones(1, 3, 384, 384)
test_input = test_input.to(device)
with torch.no_grad():
    test_output = mega(test_input)
print(test_output.shape)

torch.Size([1, 965])


### 超参数设置

In [30]:
base_lr = 8e-5

param_groups = [
    {'params': [], 'lr': base_lr * 0.125},
    {'params': [], 'lr': base_lr * 0.25},
    {'params': [], 'lr': base_lr * 0.5},
    {'params': [], 'lr': base_lr},
]

for name, param in mega.named_parameters():
    if not param.requires_grad:
        continue
    if 'layers.2.blocks' in name:
        param_groups[0]['params'].append(param)
    elif 'layers.3' in name:
        param_groups[1]['params'].append(param)
    elif name.startswith('norm'):
        param_groups[2]['params'].append(param)
    elif name.startswith('head'):
        param_groups[3]['params'].append(param)
    else:
        pass


optimizer = optim.AdamW(param_groups, weight_decay=0.01)


for i, group in enumerate(optimizer.param_groups):
    print(f'Group {i}: lr={group["lr"]}, number of parameters={len(group["params"])}')


Group 0: lr=1e-05, number of parameters=234
Group 1: lr=2e-05, number of parameters=29
Group 2: lr=4e-05, number of parameters=2
Group 3: lr=8e-05, number of parameters=2


In [31]:
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2, verbose=True,min_lr=1e-8)

## 回调

In [32]:
def create_epoch_callback(val_dataset, writer=None, batch_size=64, scheduler=None, device='cuda', 
                          num_workers=4, save_path='/kaggle/working/checkpoints', 
                          patience=5, plot_metrics=True):
    """
    epoch_callback，处理验证、调度器更新和指标记录
    
    参数:
        val_dataset: 验证数据集
        writer: tensorboard
        batch_size: 验证批次大小
        scheduler: 学习率调度器
        device: 使用的设备
        num_workers: 数据加载器工作进程数
        save_path: 模型保存路径
        patience: 早停耐心值
        plot_metrics: 是否绘制指标图
    """
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False
    )
    
    
    
    history = {
        'train_loss': [], 
        'val_loss': [], 
        'train_acc': [], 
        'val_acc': [],
        'lr': []
    }
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    def epoch_callback(trainer, epoch_data):
        nonlocal best_val_loss, patience_counter
        
        train_loss = epoch_data.get("train_loss_epoch_avg", 100)
        train_acc = epoch_data.get('train_accuracy_epoch_avg', 100)
        
        current_lr = trainer.optimizer.param_groups[3]['lr']
        history['lr'].append(current_lr)
        
        print(f"\nEpoch {trainer.epoch}/{trainer.epochs}")
        print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%")
        
        epoch = trainer.epoch-1
        
        val_loss, val_acc = validate(trainer.model, val_loader, trainer.objective, device, writer, epoch)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        if writer is not None:
            writer.add_scalar('Validation/Epoch_Loss', val_loss, epoch)
            writer.add_scalar('Validation/Epoch_Accuracy', val_acc, epoch)
        
        print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.2f}%")
        print(f"Learning Rate: {current_lr:.6f}")
        
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            
            import os
            os.makedirs(save_path, exist_ok=True)
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': trainer.model.state_dict(),
                'optimizer_state_dict': trainer.optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
                'val_acc': val_acc,
            }, f"{save_path}/best_model.pth")
            
            print(f"✅ Best model saved with val_loss: {val_loss:.4f}")
        else:
            patience_counter += 1
            if patience_counter == patience:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': trainer.model.state_dict(),
                }, f"{save_path}/early_stopped_model.pth")
                print(f"⚠️ Early stopping triggered after {patience} epochs without improvement")
                trainer.epochs = trainer.epoch
            if patience_counter >= patience:
                print(f"⚠️⚠️⚠️Should have stopped!!!")
        
        if plot_metrics and trainer.epoch % 2 == 0:
            plot_training_history(history, save_path)

        with open(f"{save_path}/training_history.txt", 'w') as f:
            for key, values in history.items():
                f.write(f'{key}: {values}\n')
    
    return epoch_callback

def validate(model, val_loader, criterion, device, writer=None, epoch=None):
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    batch_idx = 0
    
    with torch.no_grad():
        with tqdm.tqdm(val_loader, desc="Validating", leave=False) as pbar:
            for inputs, targets in pbar:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                
                loss = criterion(outputs, targets)
                val_loss += loss.item()
                
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
                
                dis_loss = loss.item()
                accuracy = 100. * correct / total
                
                if writer is not None and epoch is not None:
                    global_step = epoch * len(val_loader) + batch_idx
                    writer.add_scalar('Validation/Batch_Loss', dis_loss, global_step)
                    writer.add_scalar('Validation/Batch_Accuracy', accuracy, global_step)
                
                pbar.set_postfix(loss=f"{dis_loss:.4f}", accuracy=f"{accuracy:.2f}%")
                batch_idx += 1
    
    model.train()
    
    val_loss /= len(val_loader)
    accuracy = 100. * correct / total
    
    return val_loss, accuracy

def plot_training_history(history, save_path):
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.subplot(1, 3, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Val Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    
    plt.subplot(1, 3, 3)
    plt.plot(history['lr'])
    plt.title('Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.yscale('log')
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/training_history.png")
    plt.close()


In [33]:
epoch_callback_fn = create_epoch_callback(
    val_dataset=test,
    writer=writer,
    batch_size=16,
    scheduler=scheduler,
    device=device,
    save_path=checkpoint_out_path,
    patience=6
)

In [34]:
set_seed(training_seed)
trainer = BasicTrainer(dataset=training, model=mega, objective=criterion, batch_size=16, optimizer=optimizer, epochs=20, device=device, epoch_callback=epoch_callback_fn, writer=writer, log_interval=2)

In [35]:
trainer.train()

Epoch 0: 100%|██████████████████████| 743/743 [22:54<00:00,  1.85s/it, accuracy=25.99%, loss=3.7072]



Epoch 1/20
Train Loss: 4.7338, Train Accuracy: 25.99%


Val Loss: 3.0970, Val Accuracy: 46.99%
Learning Rate: 0.000080
✅ Best model saved with val_loss: 3.0970


Epoch 1: 100%|██████████████████████| 743/743 [22:13<00:00,  1.80s/it, accuracy=51.42%, loss=2.9831]



Epoch 2/20
Train Loss: 2.7239, Train Accuracy: 51.42%


...


Epoch 19/20
Train Loss: 0.2294, Train Accuracy: 95.74%


Val Loss: 1.6243, Val Accuracy: 73.53%
Learning Rate: 0.000008


Epoch 19: 100%|█████████████████████| 743/743 [22:35<00:00,  1.82s/it, accuracy=95.88%, loss=0.0931]



Epoch 20/20
Train Loss: 0.2193, Train Accuracy: 95.88%


Val Loss: 1.6240, Val Accuracy: 73.45%
Learning Rate: 0.000001


In [36]:
writer.close()

In [37]:
# mega.head.drop = nn.Identity()
# mega.head.fc = nn.Identity()

In [38]:
# torch.save(mega.state_dict(), 'model_n.pth')
torch.save(mega.state_dict(), '/kaggle/working/model_n.pth')

In [39]:
trainer.save("/kaggle/working", "savedb.pth")